# 02 · Hierarchical Multi-Agent Collaboration & Agent Supervision
## 动手 Demo


## 目录
| # | 内容 | 关键概念 |
|---|------|----------|
| 1 | 基础设施：模拟工具 & Trace 系统 | Trace、Audit |
| 2 | 单体 Agent 基线 | 对比参照 |
| 3 | 三层架构：Worker / Orchestrator / Supervisor | 分层职责 |
| 4 | 并行 Map-Reduce（asyncio） | 延迟优化 |
| 5 | Verifier Agent + Quality Gate | 自动校验 |
| 6 | Human-in-the-Loop（HITL） | 监督介入 |
| 7 | 完整端到端：三页报告系统 | 综合演示 |
| 8 | 安全：输入校验 + 权限矩阵 | 治理 |


## 📖 讲义

# Hierarchical Multi-Agent Collaboration
# & Agent Supervision

## 分层多 Agent 协作与监督
## 深度课程

<br>

---

## Part 1 · 基础设施：模拟工具 & Trace 系统

所有 Agent 共用的模拟 LLM / 检索工具，用 `time.sleep` 模拟真实延迟。

In [1]:
import nest_asyncio; nest_asyncio.apply()
import asyncio, time, uuid, random, json, copy
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any

CORPUS = [
    "公司差旅政策规定，国内出差需提前 3 天申请审批，国际出差需提前 10 天。",
    "所有采购金额超过 5 万元须经部门经理和 CFO 双签审批。",
    "数据安全策略要求敏感数据必须加密存储，禁止存储于个人设备。",
    "员工绩效评估每半年进行一次，需由直线经理和 HR 共同完成。",
    "供应商合同须经法务部门审核，合同期超过 1 年需 CEO 签字。",
    "信息安全事故须在发现后 24 小时内上报 CISO，并留存完整日志。",
    "远程办公政策允许每周最多 3 天居家，需提前与团队确认。",
    "知识产权归公司所有，员工在职期间的相关创作须签署归属协议。",
    "反腐败政策禁止收受任何形式的礼品，超过 100 元的礼品须上交。",
    "紧急事件响应预案要求关键系统 RTO <= 4 小时，RPO <= 1 小时。",
]

def fake_retrieve(query, top_k=5, latency=0.2):
    time.sleep(latency)
    scored = [(doc, random.uniform(0.5, 1.0)) for doc in CORPUS]
    scored.sort(key=lambda x: -x[1])
    return [{"text": doc, "score": round(s, 3)} for doc, s in scored[:top_k]]

def fake_summarize(chunks, latency=0.3):
    time.sleep(latency)
    snippet = chunks[0]["text"][:40] if chunks else "(empty)"
    return f"[Summary] Policy highlights: {snippet}... ({len(chunks)} items)"

def fake_extract_risks(chunks, latency=0.25):
    time.sleep(latency)
    keywords = ["禁止", "须", "必须", "规定", "要求"]
    risks = []
    for c in chunks[:3]:
        for kw in keywords:
            if kw in c["text"]:
                risks.append(f"Risk: violates [{c["text"][:30]}...]")
                break
    return risks or ["Risk: no obvious violations found"]

def fake_propose_measures(risks, latency=0.2):
    time.sleep(latency)
    return [f"Measure: for [{r[:25]}...], strengthen approval process" for r in risks[:3]]

def fake_nli_check(context, claim):
    time.sleep(0.05)
    return random.random() > 0.2

print("OK: simulation tools loaded")


SyntaxError: f-string: unmatched '[' (84401902.py, line 38)

### Trace 系统

全局 Trace 收集器，每个 Agent 调用都写入一条记录，支持按 `job_id` 查询完整链路。

In [ ]:
class TraceStore:
    def __init__(self):
        self._traces = []

    def record(self, job_id, agent, step, inp, out, latency_s, extra=None):
        entry = {
            "job_id":     job_id,
            "agent":      agent,
            "step":       step,
            "input":      str(inp)[:100],
            "output":     str(out)[:100],
            "latency_ms": round(latency_s * 1000),
            "timestamp":  round(time.time(), 3),
        }
        if extra:
            entry.update(extra)
        self._traces.append(entry)

    def get_job_trace(self, job_id):
        return [t for t in self._traces if t["job_id"] == job_id]

    def print_job_trace(self, job_id):
        entries = self.get_job_trace(job_id)
        print(f"\nTrace [{job_id[:8]}]  ({len(entries)} steps)")
        print("-" * 56)
        for e in entries:
            bar = "#" * min(int(e["latency_ms"] / 80), 18)
            print(f"  step {e['step']:2d} | {e['agent']:<18} | {bar:<18} {e['latency_ms']}ms")
        total = sum(e["latency_ms"] for e in entries)
        print("-" * 56)
        print(f"  Total: {len(entries)} entries, sum latency: {total}ms")

TRACE = TraceStore()
print("OK: TraceStore ready")


## 📖 讲义

# 课程全景地图

| 时间段 | Part | 内容 | 时长 |
|--------|------|------|------|
| 0:00–0:10 | 引言 | 为什么需要分层与监督？ | 10 min |
| 0:10–0:35 | Part 1 | 分层架构：三层职责与设计 | 25 min |
| 0:35–1:00 | Part 2 | 多 Agent 协作模式（五种深讲） | 25 min |
| **1:00–1:10** | ☕ | **休息** | **10 min** |
| 1:10–1:35 | Part 3 | Agent Supervision：Trace、质量门、HITL | 25 min |
| 1:35–1:55 | Part 4 | 通信机制 & 工程要点 | 20 min |
| 1:55–2:15 | Part 5 | 实战：三页报告系统（逐层拆解） | 20 min |
| **2:15–2:20** | ☕ | **休息** | **5 min** |
| 2:20–2:35 | Part 6 | 生产级要点：观测、安全、成本 | 15 min |
| 2:35–2:45 | Part 7 | 最佳实践 & 常见坑 | 10 min |
| 2:45–2:50 | Part 8 | 课堂练习导引 | 5 min |
| 2:50–3:00 | Q&A | 答疑 | 10 min |

---

---
## Part 2 · 单体 Agent 基线

所有逻辑在一个函数里串行执行，没有分层，没有 trace，没有监督。

```mermaid
flowchart LR
    User[用户] --> Mono[monolithic_agent]
    Mono --> Ans[回答]
    Mono -.-> Serial["串行: retrieve → summarize → extract_risks → propose_measures"]
```


In [ ]:
def monolithic_agent(doc_id):
    t0 = time.perf_counter()
    chunks   = fake_retrieve(f"policy {doc_id}", top_k=5, latency=0.2)
    summary  = fake_summarize(chunks, latency=0.3)
    risks    = fake_extract_risks(chunks, latency=0.25)
    measures = fake_propose_measures(risks, latency=0.2)
    elapsed  = round(time.perf_counter() - t0, 3)
    return {
        "summary": summary, "risks": risks, "measures": measures,
        "latency_s": elapsed, "has_trace": False, "supervised": False,
    }

result = monolithic_agent("policy-2025")
print(f"Latency:   {result['latency_s']}s")
print(f"Summary:   {result['summary'][:70]}")
print(f"Risks:     {len(result['risks'])} items")
print(f"\nWARN: has_trace={result['has_trace']}  supervised={result['supervised']}")
print("WARN: No trace -> cannot debug failures. No supervisor -> quality unknown.")


> **观察**：延迟 ≈ 各步叠加 (0.2+0.3+0.25+0.2 ≈ 0.95s)。出错时无法定位，质量好坏无人知晓。接下来拆分为三层架构。

## 📖 讲义

# 学习目标

学完本课，你能够：

1. **设计**分层多 Agent 架构（Worker / Orchestrator / Supervisor）
2. **选择**合适的协作模式（Pipeline、Parallel、Blackboard 等）
3. **实现** Agent Supervision：trace、质量门、HITL、审计
4. **掌握**工程要点：通信、幂等、可观测、降级、安全
5. **完成**一个包含 Orchestrator + Verifier + Supervisor 的本地演示系统

> 配套 Demo Notebook 可在课后独立运行，**无需外部依赖**

---

---

# 引言
## 为什么需要分层与监督？

---

# 从一个真实问题出发

> **任务**："为企业政策文档自动生成三页报告：
> ① 要点总结  ② 风险清单  ③ 建议措施
> 并确保内容准确、有据可查、符合法务要求"

<br>

**你会怎么设计这个系统？**

---

# 方案 A：单体 Agent

```mermaid
flowchart TB
    User[用户] --> One[one_agent document]
    One --> Report[报告]
    One --> Tools["一个 LLM + 所有工具<br/>检索 / 摘要 / 风险提取<br/>建议生成 / 事实校验 / 格式化"]
```

**问题**
- Context 窗口装不下长文档 + 全部工具调用历史
- 一步出错，整体失败，无法定位哪步错了
- 无法优化单个能力（换 Retriever 要改整个 Agent）
- 生产后：**出了问题谁负责？质量如何保证？**

---

# 方案 B：分层多 Agent + Supervision

```mermaid
flowchart TB
    Supervisor["Supervisor<br/>质量门 / HITL / 合规审计"]
    Orchestrator["Orchestrator<br/>计划 / 分派 / 汇总"]
    Retriever["Retriever Agent"]
    Summarizer["Summarizer"]
    RiskExtract["RiskExtract Agent"]
    Verifier["Verifier<br/>NLI 事实校验"]

    Supervisor --> Orchestrator
    Orchestrator -->|"并行"| Retriever
    Orchestrator -->|"并行"| Summarizer
    Orchestrator -->|"并行"| RiskExtract
    Retriever --> Verifier
    Summarizer --> Verifier
    RiskExtract --> Verifier
```

---

# 分层带来的四大收益

<div class="columns">
<div>

### 1. 职责清晰
每个组件只做一件事：
- Retriever → 只检索
- Summarizer → 只摘要
- Verifier → 只校验
- Supervisor → 只监督

**易测试、易替换、易优化**

### 2. 独立扩容
- Retriever 高并发 → 多副本
- Generator 成本高 → 精细限流
- 各自按需伸缩，互不影响

</div>
<div>

### 3. 故障隔离
- Reranker 挂了 → 降级，不影响其他
- Generator 超时 → 返回粗检结果
- **系统有弹性，不会全部崩溃**

### 4. 可观测 & 可治理
- 每步有 trace，出错可定位
- Verifier 自动校验质量
- Supervisor 记录审计日志
- HITL：高风险决策人工兜底

</div>
</div>

---


---
## Part 3 · 三层架构：Worker / Orchestrator / Supervisor

```mermaid
flowchart TB
    Supervisor["Supervisor 顶层<br/>质量门 / HITL / 审计"]
    Orchestrator["Orchestrator 中层<br/>计划 / 分派 / 汇总"]
    Retriever[RetrieverWorker]
    Summarizer[SummarizerWorker]
    Risk[RiskExtractorWorker]
    Measure[MeasureProposerWorker]

    Supervisor --> Orchestrator
    Orchestrator --> Retriever
    Orchestrator --> Summarizer
    Orchestrator --> Risk
    Orchestrator --> Measure
```


In [ ]:
# ── Worker Agents (bottom layer) ─────────────────────────────────────────
class RetrieverWorker:
    name = "retriever"
    def run(self, doc_id):
        chunks = fake_retrieve(f"policy {doc_id}", top_k=6, latency=0.2)
        return {"chunks": chunks, "count": len(chunks)}

class SummarizerWorker:
    name = "summarizer"
    def run(self, chunks):
        return {"summary": fake_summarize(chunks, latency=0.3)}

class RiskExtractorWorker:
    name = "risk_extractor"
    def run(self, chunks):
        return {"risks": fake_extract_risks(chunks, latency=0.25)}

class MeasureProposerWorker:
    name = "measure_proposer"
    def run(self, risks):
        return {"measures": fake_propose_measures(risks, latency=0.2)}


# ── Supervisor (top layer) ─────────────────────────────────────────────
class Supervisor:
    def __init__(self, min_pass_rate=0.80):
        self.min_pass_rate = min_pass_rate
        self.jobs = {}

    def start_job(self, job_id, doc_id):
        self.jobs[job_id] = {'doc_id': doc_id, 'status': 'running', 'started': time.time()}
        print(f"  [Supervisor] START job {job_id[:8]}")

    def quality_gate(self, job_id, pass_rate):
        if pass_rate >= self.min_pass_rate:
            return {'action': 'proceed',
                    'reason': f'pass_rate={pass_rate:.2f} >= {self.min_pass_rate}'}
        return {'action': 'human_review',
                'reason': f'pass_rate={pass_rate:.2f} < {self.min_pass_rate}'}

    def close_job(self, job_id, status):
        if job_id in self.jobs:
            self.jobs[job_id]['status'] = status
        print(f"  [Supervisor] CLOSE job {job_id[:8]} -> {status}")

    def create_hitl_ticket(self, job_id, reason):
        tid = f"TICKET-{job_id[:6].upper()}"
        print(f"  [Supervisor] HITL ticket: {tid}  ({reason})")
        return tid


# ── Orchestrator (middle layer) ────────────────────────────────────────
class SimpleOrchestrator:
    def __init__(self):
        self.retriever  = RetrieverWorker()
        self.summarizer = SummarizerWorker()
        self.risk_ext   = RiskExtractorWorker()
        self.measure    = MeasureProposerWorker()
        self.supervisor = Supervisor()

    def handle(self, doc_id):
        job_id = str(uuid.uuid4())
        self.supervisor.start_job(job_id, doc_id)
        step = [0]

        def run(worker, *args):
            step[0] += 1
            t = time.perf_counter()
            res = worker.run(*args)
            lat = time.perf_counter() - t
            TRACE.record(job_id, worker.name, step[0], str(args)[:40], res, lat)
            print(f"  [{worker.name:<18}] OK {round(lat,3)}s")
            return res

        ret   = run(self.retriever, doc_id)
        summ  = run(self.summarizer, ret['chunks'])
        risks = run(self.risk_ext,   ret['chunks'])
        meas  = run(self.measure,    risks['risks'])

        self.supervisor.close_job(job_id, 'passed')
        TRACE.print_job_trace(job_id)
        return {'job_id': job_id, **summ, **risks, **meas}


orc = SimpleOrchestrator()
t0 = time.perf_counter()
report = orc.handle("policy-2025")
print(f"\nTotal: {round(time.perf_counter()-t0, 3)}s")
print(f"Summary: {report['summary'][:70]}")


## 📖 讲义

# Part 1
## 分层架构：三层职责与设计

---

# 三层架构全貌

<div class="three-col">
<div>

### 顶层：Supervisor
**负责：策略 & 治理**
- 设置质量阈值
- 监听所有 trace
- 触发 Quality Gate
- 创建 HITL ticket
- 记录审计日志
- 发出告警通知

**不做**：具体任务

</div>
<div>

### 中层：Orchestrator
**负责：计划 & 协调**
- 解析用户意图
- 生成执行计划
- 分派子任务给 Worker
- 等待并收集结果
- 汇总成最终输出
- 调用 Verifier

**不做**：检索/生成/校验

</div>
<div>

### 底层：Worker Agents
**负责：单一技能**
- Retriever
- Summarizer
- RiskExtractor
- MeasureProposer
- FactChecker
- Calculator
- APIAgent

**不做**：计划/汇总/监督

</div>
</div>

> **黄金法则**：每层只做自己该做的事，通过**接口（消息/API）**交互，不直接调用内部实现

---

# Supervisor 详解

**职责**：保障整个系统的质量、可解释性与安全性

```python
class Supervisor:
    # 配置：质量阈值
    min_pass_rate   = 0.80    # Verifier pass_rate 下限
    max_latency_p95 = 5000    # ms，超出触发告警
    max_cost_per_job = 0.50   # USD，超出触发审批

    def on_job_complete(self, job_id, result):
        verdict = result["verdict"]

        # Quality Gate
        if verdict["pass_rate"] < self.min_pass_rate:
            return self.create_hitl_ticket(job_id, reason="low_quality")

        # Cost Gate
        if result["cost_usd"] > self.max_cost_per_job:
            return self.create_hitl_ticket(job_id, reason="cost_exceeded")

        # Audit
        self.write_audit_log(job_id, result, status="auto_approved")
        return {"action": "proceed"}
```

**关键**：Supervisor 只是观察者 + 决策者，**不参与具体业务逻辑**

---

# Orchestrator 详解

**职责**：接受请求 → 生成计划 → 分派 → 汇总 → 调用 Verifier

```mermaid
flowchart TB
    Start["handle_request doc_id"]
    Plan["1. plan<br/>分解为子任务列表"]
    Dispatch["2. dispatch_all<br/>asyncio.gather 并行分派"]
    Aggregate["3. aggregate<br/>合并各 Worker 结果"]
    Verify["4. verify<br/>Verifier 校验关键断言"]
    Report["5. report_to_supervisor<br/>结果 + trace 上报"]

    Start --> Plan --> Dispatch --> Aggregate --> Verify --> Report
```

**注意**：Orchestrator 可以是**静态计划**（预定义子任务）或**动态计划**（LLM 生成任务列表，适合 ReAct/LangGraph）

---

# Worker Agent 设计原则

**每个 Worker 遵循：单一职责 + 无状态 + 幂等**

```python
class WorkerAgent:
    name: str           # 唯一标识
    input_schema: dict  # 输入 JSON schema
    output_schema: dict # 输出 JSON schema

    def run(self, job: dict) -> dict:
        # 1. 校验输入 schema
        # 2. 执行单一职责
        # 3. 返回结构化输出
        # 4. 不保存任何状态到 self（无状态）
        ...
```

**可替换性示例**：

| 职责 | 实现 A | 实现 B | 切换方式 |
|------|--------|--------|----------|
| 检索 | BM25 | FAISS 向量 | 改 `retriever=` 配置 |
| 重排 | BM25 Score | CrossEncoder | 改 `reranker=` 配置 |
| 生成 | GPT-4o | Claude Sonnet | 改 `generator=` 配置 |
| 校验 | NLI Model | LLM-as-Judge | 改 `verifier=` 配置 |

---

# 分层架构的权衡

<div class="columns">
<div>

### 优势

✅ 每层独立测试，单元测试简单
✅ 可独立扩容（Retriever × N 副本）
✅ Worker 可替换，不影响其他层
✅ Trace 精确到每一步
✅ Supervisor 与业务逻辑解耦

### 适合场景
- 生产级 Agentic RAG
- 企业自动化工作流
- 内容审核 / 报告生成
- 任何需要合规审计的系统

</div>
<div>

### 代价

⚠️ 代码量更多（需定义接口）
⚠️ 调用链更长（增加网络延迟）
⚠️ 初期设计更复杂
⚠️ 调试时需要跨层追踪

### 不适合场景
- 一次性脚本 / 原型验证
- 极简单的单步任务
- 延迟要求 < 100ms 的实时系统（需优化）

> **建议**：先做单体 MVP 验证需求，再演进为分层

</div>
</div>

---

# 动手：分层架构设计练习

**场景**："智能简历筛选系统"

请在纸上或白板上画出分层架构：

1. **识别 Worker Agents**（各自的职责是什么？）
2. **定义 Orchestrator 的计划步骤**（顺序？并行？）
3. **设计 Supervisor 的质量门**（检查哪些指标？）
4. **HITL 在哪个节点触发？**

*参考答案（5 分钟后公布）*：
- Workers：ResumeParser、SkillMatcher、ExperienceScorer、CultureFitEvaluator
- Orchestrator：Parse → 并行(SkillMatch, ExperienceScore) → Aggregate → VerifyTop3
- Supervisor：匹配率 < 60% 或有争议候选 → HITL
- HITL：对最终推荐名单做人工确认，特别是 Reject 决策

---


---
## Part 4 · 并行 Map-Reduce（asyncio）

将三个独立 Worker 改为 `asyncio.gather` **并行执行**，总延迟从串行叠加变为 `max(子任务延迟)`。

In [ ]:
class ParallelOrchestrator:
    def __init__(self):
        self.retriever  = RetrieverWorker()
        self.summarizer = SummarizerWorker()
        self.risk_ext   = RiskExtractorWorker()
        self.measure    = MeasureProposerWorker()
        self.supervisor = Supervisor()

    def handle(self, doc_id):
        job_id = str(uuid.uuid4())
        self.supervisor.start_job(job_id, doc_id)

        # Stage 1: retrieve (must come first)
        t = time.perf_counter()
        ret = self.retriever.run(doc_id)
        TRACE.record(job_id, 'retriever', 1, doc_id, ret, time.perf_counter()-t)
        print(f"  [{'retriever':<18}] OK {round(time.perf_counter()-t,3)}s")

        # Stage 2: parallel workers
        async def parallel():
            loop = asyncio.get_event_loop()
            async def w(worker, args, step):
                t = time.perf_counter()
                res = await loop.run_in_executor(None, worker.run, *args)
                lat = time.perf_counter() - t
                TRACE.record(job_id, worker.name, step, str(args)[:40], res, lat)
                print(f"  [{worker.name:<18}] OK {round(lat,3)}s  (parallel)")
                return res
            return await asyncio.gather(
                w(self.summarizer, (ret['chunks'],),     step=2),
                w(self.risk_ext,   (ret['chunks'],),     step=3),
                w(self.measure,    (['riskA','riskB'],), step=4),
            )

        summ_r, risk_r, meas_r = asyncio.run(parallel())
        self.supervisor.close_job(job_id, 'passed')
        TRACE.print_job_trace(job_id)
        return {'job_id': job_id, **summ_r, **risk_r, **meas_r}


print('=== Serial vs Parallel latency comparison ===\n')
t0 = time.perf_counter()
SimpleOrchestrator().handle('doc-A')
serial_s = round(time.perf_counter() - t0, 3)

print()
t0 = time.perf_counter()
ParallelOrchestrator().handle('doc-B')
parallel_s = round(time.perf_counter() - t0, 3)

print(f'\nSerial:   {serial_s}s')
print(f'Parallel: {parallel_s}s')
print(f'Saved:    {round(serial_s - parallel_s, 3)}s  ({round((1 - parallel_s/serial_s)*100)}% improvement)')


> **关键观察**：并行使三个 Worker 延迟从**叠加**变为**取最大值**，随 Worker 数量增加，节省效果越显著。

## 📖 讲义

# Part 2
## 多 Agent 协作模式（五种深讲）

---

# 五种协作模式概览

| 模式 | 结构 | 适用场景 | 复杂度 |
|------|------|----------|--------|
| **Pipeline** | 线性链式 | 固定阶段处理 | ⭐ |
| **Parallel / Map-Reduce** | 并行 + 汇总 | 多文档、批量分析 | ⭐⭐ |
| **Orchestrator-Driven** | 中心编排 | 复杂工作流 | ⭐⭐ |
| **Blackboard** | 共享工作区 | 迭代协商 | ⭐⭐⭐ |
| **Peer-to-Peer** | 去中心化 | 自治/竞争 | ⭐⭐⭐⭐ |

> **选择建议**：从 Pipeline 起步 → 按需引入 Parallel → Orchestrator-Driven 是主力模式 → Blackboard 用于复杂迭代 → P2P 留给研究

---

# 模式 1：Pipeline（流水线）

```mermaid
flowchart LR
    Doc[文档输入] --> Retriever
    Retriever -->|chunks| Summarizer
    Summarizer -->|summary| Verifier
    Verifier --> Report[报告]
    Summarizer --> RiskExtractor
```

**特点**
- 固定阶段，顺序推进，**不可回退**（除非显式 retry）
- 每阶段输出即下一阶段输入，接口明确
- 适合：文档处理、内容审核、ETL 流水线

**实现**：Redis 队列 / Kafka Topic / Python `queue.Queue`

```python
# 简化实现
def pipeline(doc_id):
    chunks   = retriever.run(doc_id)
    summary  = summarizer.run(chunks)
    verdict  = verifier.run(summary)
    return verdict
```

**缺点**：串行延迟叠加；某阶段故障阻断整条流水线

---

# 模式 2：Parallel / Map-Reduce

```mermaid
flowchart TB
    Doc[文档] -->|split into chunks| Split
    Split --> S1[Summarizer-1]
    Split --> S2[Summarizer-2]
    Split --> S3[Summarizer-3]
    Split --> RE[RiskExtractor]
    S1 --> Agg[Aggregator]
    S2 --> Agg
    S3 --> Agg
    RE --> Agg
    Agg --> Final[final_report]
```

**关键**：Map 阶段并行，Reduce 阶段汇总
- 总延迟 ≈ `max(子任务延迟)`，而非 `sum`
- 特别适合：多文档摘要、批量实体提取

```python
# asyncio.gather 实现
async def parallel_stage(chunks):
    results = await asyncio.gather(
        summarizer.run_async(chunks[:5]),
        risk_extractor.run_async(chunks[5:10]),
        fact_checker.run_async(chunks),
    )
    return aggregator.run(results)
```

**注意**：Reduce 阶段需要处理部分失败（某个 Map 失败时的策略）

---

# 模式 2：Map-Reduce 延迟分析

```mermaid
gantt
    title 串行 vs 并行延迟对比
    dateFormat X
    axisFormat %s

    section 串行
    Task A 1.0s :a1, 0, 1
    Task B 1.0s :a2, 1, 2
    Task C 1.0s :a3, 2, 3

    section 并行
    Task A 1.0s :b1, 0, 1
    Task B 0.85s :b2, 0, 0.85
    Task C 0.95s :b3, 0, 0.95
```

**节省 = (sum - max) = (3.0 - 1.0) = 2.0s**

随着并行任务数增加，节省效果越显著：
- 10 个各 1s 的任务：串行 10s → 并行 1s（节省 90%）

---

# 模式 3：Orchestrator-Driven（主力模式）

```mermaid
flowchart TB
    User[用户] --> Orch[Orchestrator]
    Orch --> Plan["1. plan → summarize / extract_risks / propose_measures"]
    Plan --> D1["2. dispatch summarize"]
    Plan --> D2["2. dispatch extract_risks"]
    Plan --> D3["2. dispatch propose"]
    D1 --> SW[SummarizerWorker]
    D2 --> RW[RiskExtractorWorker]
    D3 --> MW[MeasureProposer]
    SW --> Agg["3. aggregate → report"]
    RW --> Agg
    MW --> Agg
    Agg --> Ver["4. verify → VerifierAgent → verdict"]
    Ver --> Sup["5. report_to_supervisor"]
```

**优势**：全程可控、每步可追踪、易加入质量门
**关键设计点**：Orchestrator 可以是**静态**（hardcoded plan）或**动态**（LLM 生成计划）

---

# 静态 vs 动态 Orchestrator

<div class="columns">
<div>

### 静态 Orchestrator
```python
# 预定义计划，固定工作流
def plan(doc_id):
    return [
        {"task": "summarize",      "depends_on": []},
        {"task": "extract_risks",  "depends_on": []},
        {"task": "propose",        "depends_on": ["extract_risks"]},
        {"task": "verify",         "depends_on": ["summarize","propose"]},
    ]
```

✅ 可预测、易测试、低成本
✅ 适合固定流程的业务
❌ 无法处理未预期的任务类型

</div>
<div>

### 动态 Orchestrator（ReAct / LangGraph）
```python
# LLM 动态生成计划
def plan(task_description):
    return llm.generate(
        system="You are a planner. Given a task,"
               "decompose it into subtasks.",
        user=task_description,
        output_schema=TaskListSchema,
    )
```

✅ 灵活，可处理新任务类型
✅ 适合开放式 Agentic 系统
❌ 成本高、不可预测、难调试
❌ 需要 LLM 质量保证

</div>
</div>

> **生产建议**：先用静态 Orchestrator 覆盖 80% 已知流程，对剩余 20% 用动态

---

# 模式 4：Blackboard（共享工作区）

```mermaid
flowchart LR
    subgraph Blackboard
        chunks["blackboard chunks"]
        hits["blackboard hits"]
        topk["blackboard topk"]
        answer["blackboard answer"]
    end

    Ingest[IngestAgent] -->|写入| chunks
    Retriever[RetrieverAgent] -->|读取| chunks
    Retriever -->|写入| hits
    Reranker[RerankerAgent] -->|读取| hits
    Reranker -->|写入| topk
    Generator[GeneratorAgent] -->|读取| topk
    Generator -->|写入| answer
    Verifier[VerifierAgent] -->|读取| answer
    chunks -.->|变更通知 Event/Pub-Sub| Supervisor
    hits -.-> Supervisor
    topk -.-> Supervisor
    answer -.-> Supervisor
```

**适合**：迭代协作、多模型协商、Agent 之间需要共享中间产物

```python
bb = Blackboard()
bb.on("chunks_ready",  retriever_agent)   # 事件驱动
bb.on("hits_ready",    reranker_agent)
bb.on("topk_ready",    generator_agent)
bb.on("answer_ready",  verifier_agent)
```

**挑战**：写入合约、冲突处理、垃圾清理（TTL）

---

# 模式 5：Peer-to-Peer / 去中心化

```mermaid
flowchart TB
    A[Agent A] <-->|negotiate| B[Agent B]
    A <-->|bid/accept| C[Agent C]
    B <-->|bid/accept| C
```

**典型场景**：
- 多个 Generator 竞价，Orchestrator 选最优结果
- 资源分配：多 Agent 协商使用共享 GPU/API 配额
- 多专家辩论：不同角色 Agent 对同一问题发表意见

**实现**：通常基于 Blackboard + 投票/竞争规则

> ⚠️ **生产慎用**：调试极难，行为难以预测，成本不可控
> ✅ **研究价值高**：适合 AI 自治系统 / 多智能体仿真

---

# 模式选择决策树

```mermaid
flowchart TB
    Q1{"任务是否有固定线性阶段？"}
    Q1 -->|YES| P[Pipeline]
    Q1 -->|NO| Q2{"是否需要并行处理多个相似子任务？"}
    Q2 -->|YES| MR[Parallel / Map-Reduce]
    Q2 -->|NO| Q3{"是否需要复杂计划与自适应？"}
    Q3 -->|YES| OD[Orchestrator-Driven 静态 or 动态]
    Q3 -->|NO| Q4{"是否需要多个 Agent 迭代共享中间产物？"}
    Q4 -->|YES| BB[Blackboard]
    Q4 -->|NO| Q5{"是否是自治系统 / 多方协商？"}
    Q5 -->|YES| P2P[Peer-to-Peer 高级]
    Q5 -->|NO| Single[重新检查需求，可能单 Agent 就够了]
```

---

# ☕ 休息 10 分钟

---

---

# Part 3
## Agent Supervision
## Trace、质量门、HITL、审计

---

# 为什么 Supervision 不可缺？

**没有 Supervision 的多 Agent 系统：**

```mermaid
flowchart LR
    User[用户] --> Orch[Orchestrator] --> Workers --> Out[输出]
```

以下问题无法回答：
- 这个答案是怎么得出的？
- 哪一步最耗时？哪一步出了错？
- 输出质量达标了吗？谁来保证？
- 用了什么模型？消耗了多少 token？
- 万一合规审计，有日志可查吗？

**Supervision 解决五类问题**

| 问题类型 | 解决手段 |
|----------|----------|
| 质量不可知 | Verifier + Quality Gate |
| 故障无法定位 | 全链路 Trace |
| 高风险决策 | Human-in-the-Loop |
| 合规 / 法务审计 | Immutable Audit Log |
| 成本 / 安全失控 | Policy + Budget Gate |

---

# Trace 设计：捕获什么？

**最小 Trace Entry**（每个 Agent 每次调用产生一条）：

```json
{
  "trace_id":   "t-uuid",
  "job_id":     "j-uuid",
  "agent":      "summarizer",
  "step":       3,
  "input":  { "chunks": ["..."], "top_k": 5 },
  "output": { "summary": "..." },
  "start_time": "2025-01-15T10:30:00.000Z",
  "end_time":   "2025-01-15T10:30:01.200Z",
  "metrics": {
    "latency_ms": 1200,
    "tokens_in":  850,
    "tokens_out": 320,
    "model":      "claude-haiku-4-5",
    "cost_usd":   0.0012
  },
  "parent_trace_id": "t-uuid-orchestrator"
}
```

**关键字段**：`job_id`（全链路聚合）、`parent_trace_id`（层级关系）、`metrics`（性能与成本）

---

# Trace 层级：从 Job 到 Step

```mermaid
flowchart TB
    Job["job_id: j-001"]
    Orch["trace: orchestrator step 0<br/>latency=2.5s"]
    Ret["trace: retriever step 1<br/>0.3s"]
    Sum["trace: summarizer step 2<br/>1.2s"]
    Risk["trace: risk_extractor step 3<br/>0.9s"]
    Meas["trace: measure_proposer step 4<br/>0.5s"]
    Ver["trace: verifier step 5<br/>0.6s"]
    C1["nli_check claim_1 entailed=True"]
    C2["nli_check claim_2 entailed=False"]
    C3["nli_check claim_3 entailed=True"]
    Gate["quality_gate pass_rate=0.67<br/>action=human_review"]

    Job --> Orch
    Orch --> Ret
    Orch --> Sum
    Orch --> Risk
    Orch --> Meas
    Orch --> Ver
    Ver --> C1
    Ver --> C2
    Ver --> C3
    Job --> Gate
```

**生产工具**：
- **Langfuse** / **LangSmith** — LLM 专用 trace，支持 prompt/token 记录
- **OpenTelemetry + Jaeger** — 标准分布式追踪，适合微服务
- **自研 TraceStore** — 教学/轻量场景（即本课 Demo）

---

# Quality Gate：量化质量

```python
@dataclass
class QualityConfig:
    min_pass_rate:   float = 0.80   # Verifier NLI pass rate
    min_faithfulness: float = 0.75  # RAGAS faithfulness
    max_latency_p95: int   = 5000   # ms
    max_cost_per_job: float = 0.50  # USD

def quality_gate(result: dict, cfg: QualityConfig) -> dict:
    checks = {
        "verifier_pass": result["pass_rate"] >= cfg.min_pass_rate,
        "faithfulness":  result.get("faithfulness", 1.0) >= cfg.min_faithfulness,
        "latency_ok":    result["latency_ms"] <= cfg.max_latency_p95,
        "cost_ok":       result.get("cost_usd", 0) <= cfg.max_cost_per_job,
    }
    n_failed = sum(1 for v in checks.values() if not v)

    if n_failed == 0:   return {"action": "proceed",       "level": "INFO"}
    if n_failed == 1:   return {"action": "flag_for_review","level": "WARN"}
    return              {"action": "human_review",          "level": "BLOCK"}
```

---

# HITL（Human-in-the-Loop）分级

| 级别 | 触发条件 | 处理方式 | 等待人工？ |
|------|----------|----------|-----------|
| **INFO** | 全部指标正常 | 自动通过，记录 trace | ❌ |
| **WARN** | 1 项指标低于阈值 | 标注存疑，后台异步复核 | ❌（非阻塞） |
| **REVIEW** | 2+ 项低或高风险 | 创建 ticket，等待审核 | ✅（阻塞） |
| **BLOCK** | 安全规则 / PII / 法务 | 立即停止，通知安全团队 | ✅（强制） |

**HITL 工作流**：

```mermaid
flowchart TB
    T["Supervisor 创建 ticket<br/>含 trace / QA 报告 / Verifier 细节"]
    UI["审核界面 Slack / 内部工具 / 邮件<br/>审核人看到：是什么、为什么低质量、建议操作"]
    Dec["审核人决策：批准 / 拒绝 / 要求修改"]
    Act["Orchestrator 继续 / 回滚 / 触发重新生成"]

    T --> UI --> Dec --> Act
```

---

# HITL 实现要点

<div class="columns">
<div>

### 挂起与恢复
```python
# 挂起 job
def suspend_job(job_id, ticket_id):
    db.update(job_id, status="suspended",
              ticket_id=ticket_id)
    # 保存完整上下文（供恢复使用）
    ctx_store.save(job_id, full_context)

# 人工决策后恢复
def resume_job(job_id, decision):
    ctx = ctx_store.load(job_id)
    if decision == "approve":
        continue_from_checkpoint(ctx)
    elif decision == "revise":
        re_run_with_feedback(ctx,
            feedback=decision["notes"])
    else:  # reject
        mark_failed(job_id)
```

</div>
<div>

### 给审核人的信息
```json
{
  "ticket_id": "TICKET-ABC123",
  "summary":   "3 页政策报告草稿",
  "quality": {
    "pass_rate":    0.67,
    "failed_claims": [
      "风险条款 2 无文档支撑",
      "建议措施 1 与策略矛盾"
    ]
  },
  "trace_url": "https://langfuse/job/abc123",
  "suggested_action": "REVISE",
  "auto_deadline": "2025-01-16T10:00:00Z"
}
```

**关键**：给审核人足够信息，让决策有据可查

</div>
</div>

---

# Audit Log：合规与回溯

**不可变审计日志**（每条决策都必须记录）：

```json
{
  "event_id":    "audit-uuid",
  "job_id":      "j-uuid",
  "event_type":  "human_review_decision",
  "timestamp":   "2025-01-15T11:30:00Z",
  "actor":       "alice@company.com",
  "decision":    "approve",
  "reason":      "Content verified by domain expert",
  "overrides": {
    "original_pass_rate": 0.67,
    "approved_despite_low_quality": true
  },
  "ip_address":  "10.0.0.1",
  "session_id":  "sess-uuid"
}
```

**存储要求**：不可修改（Append-only）、加密、访问控制、保留期（通常 5-7 年）

**工具**：AWS CloudTrail、Elasticsearch with ILM、PostgreSQL with audit trigger

---


---
## Part 5 · Verifier Agent + Quality Gate

Verifier 对报告中的关键断言做 NLI 校验，Quality Gate 根据 `pass_rate` 决定是否放行。

In [ ]:
class VerifierAgent:
    name = 'verifier'

    def verify(self, report, job_id):
        claims = self._extract_claims(report)
        step_n = len(TRACE.get_job_trace(job_id)) + 1
        results = []
        for claim in claims:
            context   = random.choice(CORPUS)
            supported = fake_nli_check(context, claim)
            results.append({'claim': claim[:50], 'supported': supported})
        pass_rate = sum(r['supported'] for r in results) / max(len(results), 1)
        verdict = {
            'pass':           pass_rate >= 0.80,
            'pass_rate':      round(pass_rate, 3),
            'claims_checked': len(results),
            'details':        results,
        }
        t = time.perf_counter()
        time.sleep(0.05 * len(claims))
        TRACE.record(job_id, self.name, step_n, 'report', verdict,
                     time.perf_counter()-t, extra={'pass_rate': pass_rate})
        return verdict

    def _extract_claims(self, report):
        claims = []
        if 'summary' in report:
            claims.append(f"Summary: {report['summary'][:40]}")
        for r in report.get('risks', [])[:3]:
            claims.append(r[:40])
        return claims or ['Content complies with company policy']


_verifier = VerifierAgent()
_sup      = Supervisor(min_pass_rate=0.80)

def orchestrate_with_verify(doc_id, force_fail=False):
    job_id = str(uuid.uuid4())
    _sup.start_job(job_id, doc_id)
    chunks   = fake_retrieve(f'policy {doc_id}', top_k=5, latency=0.2)
    summary  = fake_summarize(chunks, latency=0.2)
    risks    = fake_extract_risks(chunks, latency=0.2)
    measures = fake_propose_measures(risks, latency=0.15)
    TRACE.record(job_id, 'workers', 1, doc_id, 'done', 0.75)
    report = {'summary': summary, 'risks': risks, 'measures': measures}

    if force_fail:
        verdict = {'pass': False, 'pass_rate': 0.33, 'claims_checked': 3, 'details': []}
        TRACE.record(job_id, 'verifier', 2, 'report', verdict, 0.15,
                     extra={'pass_rate': 0.33})
        print(f'  [verifier          ] WARN pass_rate=0.33 (forced low quality)')
    else:
        print(f'  [verifier          ]', end=' ')
        verdict = _verifier.verify(report, job_id)
        icon = 'OK' if verdict['pass'] else 'WARN'
        print(f"{icon} pass_rate={verdict['pass_rate']}  ({verdict['claims_checked']} claims)")

    gate = _sup.quality_gate(job_id, verdict['pass_rate'])
    print(f"  [quality_gate      ] action={gate['action']}  ({gate['reason']})")

    if gate['action'] == 'human_review':
        ticket = _sup.create_hitl_ticket(job_id, gate['reason'])
        _sup.close_job(job_id, 'under_review')
        return {'status': 'under_review', 'ticket_id': ticket, 'job_id': job_id}

    _sup.close_job(job_id, 'passed')
    return {'status': 'ok', 'report': report, 'verdict': verdict, 'job_id': job_id}


print('=== Scenario A: normal quality (should pass) ===')
r = orchestrate_with_verify('policy-2025')
print(f"Result: {r['status']}\n")

print('=== Scenario B: low quality (should trigger HITL) ===')
r = orchestrate_with_verify('policy-2025', force_fail=True)
print(f"Result: {r['status']}  ticket={r.get('ticket_id')}")


## 📖 讲义

# Part 4
## 通信机制 & 工程要点

---

# 四种通信机制深度对比

| 机制 | 延迟 | 吞吐 | 耦合 | 容错 | 典型工具 |
|------|------|------|------|------|----------|
| **Sync HTTP** | 低 | 中 | 高 | 弱 | FastAPI, Flask |
| **Async gRPC** | 很低 | 高 | 中 | 中 | gRPC streaming |
| **Message Queue** | 中 | 很高 | 低 | 强 | Redis, Kafka, SQS |
| **Pub-Sub** | 中 | 很高 | 很低 | 强 | Kafka, SNS, Redis |

**选择原则**：

```mermaid
flowchart TB
    Q1{"用户直接等待结果？"}
    Q1 -->|YES| Sync[Sync HTTP / gRPC]
    Q1 -->|NO| Q2{"任务耗时 > 1s 或需要重试？"}
    Q2 -->|YES| MQ[Message Queue]
    Q2 -->|NO| Q3{"广播事件给多个消费者？"}
    Q3 -->|YES| PS[Pub-Sub]
    Q3 -->|NO| Q4{"大文件 / artifacts 传递？"}
    Q4 -->|YES| S3[Shared Storage S3 + 信号]
```

---

# 消息队列深入：幂等 + 重试

**问题**：网络故障或消费者崩溃后，消息会**重复投递**

```mermaid
flowchart LR
    Producer --> Queue
    Queue --> C1["Consumer 处理中崩溃"]
    Queue --> C2["Consumer 重试：再次处理"]
    C2 -.->|没有幂等| Risk["重复写入 DB / 重复调用外部 API"]
```

**解决：幂等键（Idempotency Key）**

```python
def process_job(job: dict):
    rid = job["request_id"]   # 唯一键：job_id + task_name

    # 幂等检查（Redis SET / DB 唯一索引）
    if redis.get(f"processed:{rid}"):
        logger.info(f"Skip duplicate: {rid}")
        return

    # 业务逻辑
    result = do_work(job)

    # 原子写入：结果 + 幂等标记
    with db.transaction():
        db.save_result(rid, result)
        redis.setex(f"processed:{rid}", 86400, "1")  # TTL 24h
```

---

# 重试策略：Exponential Backoff + Jitter

```python
import random, time

def retry_with_backoff(fn, max_retries=3, base_delay=1.0):
    for attempt in range(max_retries + 1):
        try:
            return fn()
        except TransientError as e:
            if attempt == max_retries:
                raise   # 超过最大重试，走降级路径

            # Exponential backoff + jitter（防止惊群效应）
            delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
            logger.warning(f"Retry {attempt+1}/{max_retries} in {delay:.2f}s: {e}")
            time.sleep(delay)
        except PermanentError:
            raise   # 永久错误不重试（如输入校验失败）

# 延迟序列（base=1s）：
# attempt 0: 立即
# attempt 1: 1~2s
# attempt 2: 2~4s
# attempt 3: 4~8s（超出则走降级）
```

---

# 状态管理：短期 vs 长期

<div class="columns">
<div>

### 短期状态（任务内）
随消息一起传递，不持久化

```python
job = {
    "request_id": "uuid",
    "doc_id":     "policy-2025",
    "query":      "生成三页报告",
    # 阶段产物随消息传递
    "chunks":     [...],
    "summary":    "...",
    "risks":      [...],
}
```

生命周期：随 job 完成而消失
存储：内存 / 消息体本身

</div>
<div>

### 长期状态（跨会话）
存储在持久化存储，有生命周期管理

```python
# 用户偏好、历史摘要缓存
class LongTermStore:
    def save(self, user_id, key, value, ttl_days=30):
        redis.setex(
            f"user:{user_id}:{key}",
            ttl_days * 86400,
            json.dumps(value)
        )

    def get(self, user_id, key):
        raw = redis.get(f"user:{user_id}:{key}")
        return json.loads(raw) if raw else None
```

生命周期：由 TTL 或业务规则控制
存储：Redis / PostgreSQL / DynamoDB

</div>
</div>

---

# Backpressure & 流量控制

```mermaid
flowchart TB
    P["Producer 提交速率 100 req/s"] --> Q["Queue 缓冲区"]
    Q -->|"Consumer 仅 10 req/s → 积压 → OOM"| C["Consumer 处理速率 10 req/s"]
```

**四种控制手段**

| 手段 | 实现 | 效果 |
|------|------|------|
| 队列长度上限 | `maxsize=1000`（Python Queue） | 超出时 produce 阻塞 |
| 速率限制 | Token Bucket（Redis lua script） | 平滑入流量 |
| 并发数限制 | `asyncio.Semaphore(10)` | 控制 Worker 并发 |
| 背压信号 | 队列 > 80% 时返回 HTTP 429 | 快速失败，上游降速 |

```python
# Redis Token Bucket（生产可用）
def is_allowed(key, rate=10, burst=20):
    """rate=10 req/s, burst=20 最大突发"""
    now = time.time()
    # Lua 脚本保证原子性（省略完整实现）
    ...
```

---


---
## Part 6 · Human-in-the-Loop（HITL）详细模拟

完整模拟：创建 ticket → 人工审核（批准/拒绝/修订）→ 审计日志。

In [ ]:
from enum import Enum

class HITLDecision(Enum):
    APPROVE = 'approve'
    REJECT  = 'reject'
    REVISE  = 'revise'

@dataclass
class HITLTicket:
    ticket_id:   str
    job_id:      str
    reason:      str
    report:      dict
    verdict:     dict
    status:      str               = 'pending'
    decision:    object            = None
    reviewer:    str               = ''
    reviewed_at: float             = 0.0
    notes:       str               = ''


class HITLSystem:
    def __init__(self):
        self.tickets = {}

    def create_ticket(self, job_id, report, verdict, reason):
        tid = f'TICKET-{job_id[:6].upper()}'
        t   = HITLTicket(tid, job_id, reason, report, verdict)
        self.tickets[tid] = t
        print(f'\n  HITL Ticket: {tid}')
        print(f"    reason:    {reason}")
        print(f"    pass_rate: {verdict.get('pass_rate','N/A')}")
        return t

    def review(self, ticket_id, decision, reviewer='reviewer@company.com', notes=''):
        t = self.tickets[ticket_id]
        t.status      = 'reviewed'
        t.decision    = decision
        t.reviewer    = reviewer
        t.reviewed_at = time.time()
        t.notes       = notes
        icons = {'approve': 'APPROVED', 'reject': 'REJECTED', 'revise': 'REVISE'}
        print(f"\n  [{icons[decision.value]}] {reviewer}: {decision.value}")
        if notes:
            print(f'    notes: {notes}')
        return t

    def audit_log(self):
        return [{
            'ticket_id': t.ticket_id,
            'job_id':    t.job_id,
            'status':    t.status,
            'decision':  t.decision.value if t.decision else None,
            'reviewer':  t.reviewer,
            'pass_rate': t.verdict.get('pass_rate'),
            'reviewed_at': round(t.reviewed_at, 2),
        } for t in self.tickets.values()]


hitl   = HITLSystem()
job_id = str(uuid.uuid4())

# 1. System triggers HITL (low quality)
low_verdict = {'pass': False, 'pass_rate': 0.50, 'claims_checked': 4, 'details': []}
ticket = hitl.create_ticket(job_id, {'summary': 'draft'}, low_verdict,
                             'pass_rate=0.50 < 0.80')

# 2. Human reviews: APPROVE
print('\n--- Human review ---')
hitl.review(ticket.ticket_id, HITLDecision.APPROVE,
            reviewer='alice@company.com',
            notes='Content acceptable; please improve verifier quality.')

# 3. Audit log
print('\nAudit Log:')
print(json.dumps(hitl.audit_log(), ensure_ascii=False, indent=2))

# 4. REJECT scenario
job_id2 = str(uuid.uuid4())
bad = {'pass': False, 'pass_rate': 0.20, 'claims_checked': 5}
t2  = hitl.create_ticket(job_id2, {}, bad, 'pass_rate extremely low')
hitl.review(t2.ticket_id, HITLDecision.REJECT, reviewer='bob@company.com',
            notes='Report seriously incorrect, must regenerate.')
print(f'\nTotal audit entries: {len(hitl.audit_log())}')


## 📖 讲义

# Part 5
## 实战：三页报告系统
## 逐层拆解

---

# 系统需求回顾

> **输入**：企业政策文档 ID
> **输出**：三页结构化报告
> - 第一页：要点总结
> - 第二页：风险清单
> - 第三页：建议措施
>
> **要求**：Verifier 校验关键断言；pass_rate < 0.8 触发 HITL；全程 trace；有输入校验

**架构决策**

| 组件 | 选型理由 |
|------|----------|
| Orchestrator-Driven | 固定三任务，静态计划，可预测 |
| asyncio.gather | 三个 Worker 互相独立，适合并行 |
| Verifier + Quality Gate | 法务要求内容准确性 |
| HITL（REVIEW 级） | pass_rate < 0.8 时阻塞等待人工 |
| TraceStore | 教学场景，轻量内存存储 |

---

# 完整架构图

```mermaid
flowchart TB
    User["用户 POST /generate_report<br/>doc_id: policy-2025"]
    Supervisor["Supervisor<br/>start_job / close_job<br/>quality_gate / create_hitl_ticket"]
    Orchestrator["Orchestrator<br/>1. retriever.run<br/>2. asyncio.gather summarizer/risk/measure<br/>3. aggregate<br/>4. verifier.verify"]
    Retriever["Retriever 0.2s"]
    Summarizer["Summarizer 0.3s"]
    RiskExt["RiskExt 0.25s"]
    Measure["MeasureProposer 0.2s"]
    Verifier["Verifier NLI校验"]

    User --> Supervisor
    Supervisor -->|监督| Orchestrator
    Orchestrator --> Retriever
    Orchestrator --> Summarizer
    Orchestrator --> RiskExt
    Orchestrator --> Measure
    Orchestrator --> Verifier
```

---

# Orchestrator 核心代码

```python
class ReportOrchestrator:
    def generate(self, doc_id: str) -> dict:
        job_id = str(uuid.uuid4())
        self.supervisor.start_job(job_id, doc_id)

        # Stage 1: 检索（其他阶段依赖此结果）
        ret = self._timed_run("retriever", self.retriever, doc_id, job_id, step=1)

        # Stage 2: 并行执行三个独立 Worker
        async def parallel():
            loop = asyncio.get_event_loop()
            async def w(name, worker, args, step):
                t = time.perf_counter()
                res = await loop.run_in_executor(None, worker.run, *args)
                lat = time.perf_counter() - t
                TRACE.record(job_id, name, step, str(args)[:40], res, lat)
                return res
            return await asyncio.gather(
                w("summarizer",  self.summarizer,  (ret["chunks"],), step=2),
                w("risk_ext",    self.risk_ext,    (ret["chunks"],), step=3),
                w("measure",     self.measure,     (["r1","r2"],),   step=4),
            )
        summ, risks, meas = asyncio.run(parallel())

        # Stage 3: 汇总 + Verify + Quality Gate
        report  = self._aggregate(summ, risks, meas)
        verdict = self.verifier.verify(report, job_id)
        gate    = self.supervisor.quality_gate(job_id, verdict["pass_rate"])
        return self._finalize(job_id, report, verdict, gate)
```

---

# Verifier 实现详解

```python
class VerifierAgent:
    """对报告关键断言做 NLI 校验"""

    def verify(self, report: dict, job_id: str) -> dict:
        # 1. 从报告中提取可验证断言
        claims = self._extract_claims(report)
        # 例：["政策要求国际出差提前 10 天申请", "违反数据安全策略需 24h 上报"]

        results = []
        for claim in claims:
            # 2. 检索支撑文本
            context = self._retrieve_support(claim)

            # 3. NLI 判断：context 是否蕴含 claim
            #    生产：cross-encoder/nli-deberta-v3-small 或 LLM-as-Judge
            supported = self._nli_entailment(context, claim)

            results.append({
                "claim":     claim,
                "supported": supported,
                "context":   context[:100],
                "confidence": 0.9 if supported else 0.3,
            })

        pass_rate = sum(r["supported"] for r in results) / max(len(results), 1)
        return {
            "pass":      pass_rate >= 0.80,
            "pass_rate": round(pass_rate, 3),
            "details":   results,
        }
```

---

# 运行示例：正常路径

```
[Supervisor] START job a1b2c3d4
  [retriever         ] OK 0.201s   6 chunks
  [summarizer        ] OK 0.302s   (parallel)
  [risk_extractor    ] OK 0.251s   (parallel)
  [measure_proposer  ] OK 0.203s   (parallel)
  [verifier          ] OK pass_rate=0.917  (12 claims)
  [quality_gate      ] proceed  (pass_rate=0.917 >= 0.80)
[Supervisor] CLOSE job a1b2c3d4 -> passed

Trace [a1b2c3d4]  (5 steps)
────────────────────────────────────────────────────────
  step 1 | retriever          | ##          201ms
  step 2 | summarizer         | ###         302ms  (parallel)
  step 3 | risk_extractor     | ###         251ms  (parallel)
  step 4 | measure_proposer   | ##          203ms  (parallel)
  step 5 | verifier           | #           112ms
────────────────────────────────────────────────────────
  Total: 5 entries  wall clock: 0.609s  (vs serial 0.959s)
```

**并行节省**：`(0.302 + 0.251 + 0.203) - max(0.302, 0.251, 0.203) = 0.756 - 0.302 = 0.454s`

---

# 运行示例：HITL 触发路径

```
[Supervisor] START job e5f6g7h8
  [retriever         ] OK 0.201s   6 chunks
  [summarizer        ] OK 0.305s   (parallel)
  [risk_extractor    ] OK 0.249s   (parallel)
  [measure_proposer  ] OK 0.202s   (parallel)
  [verifier          ] WARN pass_rate=0.583  (12 claims)
    ↳ failed claims:
       [2] "措施 3 与现行法规矛盾" → NOT supported
       [5] "风险 2 已有解决方案"   → NOT supported
       [7] "建议措施 1 成本合理"   → NOT supported
       [9] "引用条款 7.2.3"        → NOT supported
       [11] "数据保留期 7 年"      → NOT supported
  [quality_gate      ] human_review  (pass_rate=0.583 < 0.80)
  [Supervisor] HITL ticket: TICKET-E5F6G7
                 → Slack/Email 通知审核人
[Supervisor] CLOSE job e5f6g7h8 -> under_review

返回给用户：{"status": "under_review", "ticket_id": "TICKET-E5F6G7"}
```

---

# ☕ 休息 5 分钟

---

---

# Part 6
## 生产级要点
## 观测、安全、成本

---

# 可观测性：三大支柱

<div class="three-col">
<div>

### Metrics（指标）
```python
# 每个 Agent 暴露
from prometheus_client import (
    Counter, Histogram
)

req_total = Counter(
    "agent_requests_total",
    "Requests",
    ["agent", "status"]
)
latency = Histogram(
    "agent_latency_seconds",
    "Latency",
    ["agent"]
)

# 使用
with latency.labels("summarizer").time():
    result = summarizer.run(chunks)
req_total.labels(
    "summarizer", "ok"
).inc()
```

</div>
<div>

### Traces（链路）
```python
from opentelemetry import trace

tracer = trace.get_tracer("summarizer")

with tracer.start_as_current_span(
    "summarize_chunks"
) as span:
    span.set_attribute("chunk_count",
                       len(chunks))
    span.set_attribute("model",
                       "claude-haiku")
    result = llm.generate(...)
    span.set_attribute(
        "tokens_used",
        result["usage"]["total"]
    )
```

</div>
<div>

### Logs（日志）
```python
import structlog
log = structlog.get_logger()

log.info(
    "agent_completed",
    job_id=job_id,
    agent="summarizer",
    latency_ms=elapsed_ms,
    pass_rate=verdict["pass_rate"],
    tokens_used=usage["total"],
    model="claude-haiku-4-5",
    # 绝不记录：
    # raw_prompt / raw_output
    # user PII
)
```

</div>
</div>

> **生产工具**：Prometheus + Grafana（Metrics）、Langfuse / Jaeger（Traces）、ELK / Datadog（Logs）

---

# 安全：多层防御

```mermaid
flowchart TB
    In[用户输入]
    V1["① Input Validation<br/>拦截恶意字段 / 过长输入 / 非白名单"]
    V2["② Authentication & Authorization<br/>确认身份与 doc_id 权限"]
    V3["③ PII Detection & Redaction<br/>脱敏手机号 / 身份证 / 银行卡"]
    V4["④ Prompt Injection Guard<br/>检测 ignore previous instructions 等"]
    W[Workers LLM 调用]
    V5["⑤ Output Filtering<br/>过滤 PII / 有害内容 / 敏感信息"]
    V6["⑥ Audit Log<br/>记录调用者 / 时间 / 输入输出摘要"]

    In --> V1 --> V2 --> V3 --> V4 --> W --> V5 --> V6
```

---

# 权限矩阵：最小权限原则

| Agent | 读文档 | 写DB | 调LLM | 发告警 | 读审计日志 |
|-------|--------|------|-------|--------|-----------|
| Retriever | ✅ | ❌ | ❌ | ❌ | ❌ |
| Summarizer | ✅ | ❌ | ✅ | ❌ | ❌ |
| RiskExtractor | ✅ | ❌ | ✅ | ❌ | ❌ |
| Verifier | ✅ | ❌ | ✅ | ❌ | ❌ |
| Orchestrator | ✅ | ✅(job) | ❌ | ❌ | ❌ |
| Supervisor | ✅ | ✅(audit) | ❌ | ✅ | ✅ |

**实现方式**：
- **IAM Role**（AWS/GCP）：每个 Agent 独立 Service Account
- **API Key 分离**：不同 Agent 用不同的 LLM API Key，设置独立限额
- **网络策略**（Kubernetes NetworkPolicy）：Worker 之间不能直接互通

---

# 成本控制：三层策略

<div class="columns">
<div>

### 模型分层路由
```python
MODEL_ROUTING = {
    "retrieval_score":   "haiku-4-5",
    "rerank_score":      "haiku-4-5",
    "summarize_short":   "haiku-4-5",
    "summarize_long":    "sonnet-4-6",
    "risk_extract":      "sonnet-4-6",
    "generate_complex":  "opus-4-6",
    "nli_verify":        "haiku-4-5",
}

def get_model(task_type):
    return MODEL_ROUTING.get(
        task_type, "sonnet-4-6"
    )
```

Haiku ≈ 3× cheaper than Sonnet
Sonnet ≈ 5× cheaper than Opus

</div>
<div>

### 语义缓存
```python
import hashlib

def cached_llm_call(prompt, model):
    key = hashlib.sha256(
        f"{model}:{prompt}".encode()
    ).hexdigest()
    cached = redis.get(f"llm:{key}")
    if cached:
        return json.loads(cached)
    result = llm.call(prompt, model)
    redis.setex(f"llm:{key}", 3600,
                json.dumps(result))
    return result
```

### 预算门
```python
if job_cost_estimate > MAX_COST:
    return {"status": "cost_exceeded",
            "requires_approval": True}
```

</div>
</div>

---

# Graceful Degradation（优雅降级）

```python
async def generate_with_fallback(doc_id: str) -> dict:
    """三级降级策略"""
    try:
        # Level 0: 完整路径
        ret  = await retriever.run_async(doc_id)
        topk = await reranker.run_async(ret)     # 可能失败
        ans  = await generator.run_async(topk)
        return {"answer": ans, "quality": "full"}

    except RerankerUnavailable:
        # Level 1: 跳过 Reranker，用粗检结果
        logger.warning("reranker_down", doc_id=doc_id)
        ans = await generator.run_async(ret["hits"][:5])
        return {"answer": ans, "quality": "degraded_no_rerank"}

    except GeneratorTimeout:
        # Level 2: 直接返回检索片段
        return {"answer": None,
                "snippets": ret["hits"][:3],
                "quality": "degraded_snippets_only"}

    except Exception:
        # Level 3: 静态兜底
        return {"answer": "系统暂时不可用，请稍后重试。",
                "quality": "fallback_static"}
```

---


---
## Part 7 · 完整端到端：三页报告生成系统

把所有组件组合：并行 Workers + Verifier + Supervisor + HITL。

In [ ]:
class ThreePageReportSystem:
    def __init__(self, min_pass_rate=0.80):
        self.retriever  = RetrieverWorker()
        self.summarizer = SummarizerWorker()
        self.risk_ext   = RiskExtractorWorker()
        self.measure    = MeasureProposerWorker()
        self.verifier   = VerifierAgent()
        self.supervisor = Supervisor(min_pass_rate=min_pass_rate)
        self.hitl       = HITLSystem()

    def generate_report(self, doc_id):
        job_id  = str(uuid.uuid4())
        t_total = time.perf_counter()
        self.supervisor.start_job(job_id, doc_id)

        # Stage 1: retrieve
        t   = time.perf_counter()
        ret = self.retriever.run(doc_id)
        TRACE.record(job_id, 'retriever', 1, doc_id, ret, time.perf_counter()-t)
        print(f"  [{'retriever':<18}] OK {round(time.perf_counter()-t,3)}s  {ret['count']} chunks")

        # Stage 2: parallel workers
        async def parallel():
            loop = asyncio.get_event_loop()
            async def w(worker, args, step):
                t = time.perf_counter()
                res = await loop.run_in_executor(None, worker.run, *args)
                lat = time.perf_counter() - t
                TRACE.record(job_id, worker.name, step, str(args)[:40], res, lat)
                print(f"  [{worker.name:<18}] OK {round(lat,3)}s  (parallel)")
                return res
            return await asyncio.gather(
                w(self.summarizer, (ret['chunks'],),     step=2),
                w(self.risk_ext,   (ret['chunks'],),     step=3),
                w(self.measure,    (['riskA','riskB'],), step=4),
            )

        summ_r, risk_r, meas_r = asyncio.run(parallel())
        report = {
            'page1_summary':  summ_r['summary'],
            'page2_risks':    risk_r['risks'],
            'page3_measures': meas_r['measures'],
        }

        # Stage 3: verify
        print(f'  [verifier          ]', end=' ')
        verdict = self.verifier.verify(report, job_id)
        icon = 'OK' if verdict['pass'] else 'WARN'
        print(f"{icon} pass_rate={verdict['pass_rate']}  ({verdict['claims_checked']} claims)")

        # Stage 4: quality gate
        gate = self.supervisor.quality_gate(job_id, verdict['pass_rate'])
        print(f"  [quality_gate      ] {gate['action']}  ({gate['reason']})")

        if gate['action'] == 'human_review':
            ticket = self.hitl.create_ticket(job_id, report, verdict, gate['reason'])
            self.supervisor.close_job(job_id, 'under_review')
            TRACE.print_job_trace(job_id)
            return {'status': 'under_review', 'ticket_id': ticket.ticket_id,
                    'job_id': job_id, 'latency_s': round(time.perf_counter()-t_total,3)}

        self.supervisor.close_job(job_id, 'passed')
        TRACE.print_job_trace(job_id)
        return {'status': 'ok', 'report': report, 'verdict': verdict,
                'job_id': job_id, 'latency_s': round(time.perf_counter()-t_total,3)}


print('=' * 60)
print('Three-Page Report System - End-to-End Run')
print('=' * 60)
system = ThreePageReportSystem(min_pass_rate=0.80)
result = system.generate_report('policy-2025')

print(f"\nStatus:  {result['status']}")
print(f"Latency: {result['latency_s']}s")
if result['status'] == 'ok':
    r = result['report']
    print(f"\nPage 1 (Summary): {r['page1_summary'][:80]}")
    print('\nPage 2 (Risks):')
    for risk in r['page2_risks'][:3]:
        print(f'  - {risk[:65]}')
    print('\nPage 3 (Measures):')
    for m in r['page3_measures'][:3]:
        print(f'  - {m[:65]}')


## 📖 讲义

# Part 7
## 最佳实践 & 常见坑

---

# 最佳实践 Top 10

| # | 实践 | 说明 |
|---|------|------|
| 1 | **Plan → Dispatch → Aggregate** | Orchestrator 三步法，不做具体任务 |
| 2 | **结构化消息 + JSON Schema** | 每个任务/结果都有 schema，接口稳定 |
| 3 | **幂等 + request_id** | 每条消息带唯一键，重试安全 |
| 4 | **每步必有 Trace** | job_id 串联，latency/tokens/model 全记 |
| 5 | **Quality Gate + 分级 HITL** | 自动校验，低置信度才转人工 |
| 6 | **并行化独立子任务** | `asyncio.gather` 减少总延迟 |
| 7 | **Worker 最小权限** | 每个 Worker 只有完成本职所需权限 |
| 8 | **模型分层路由** | Haiku/Sonnet/Opus 按任务复杂度选 |
| 9 | **重试 + 降级 + HITL** | 三级故障策略，不静默失败 |
| 10 | **渐进部署** | 串行 MVP → 并行 → 加 Verifier → 加 Supervisor |

---

# 常见坑速查表

| 坑 | 典型症状 | 根因 | 对策 |
|----|----------|------|------|
| Orchestrator 单点瓶颈 | Orch CPU 100%，Worker 空转 | 逻辑都堆在 Orch | 拆分调度，引入消息队列 |
| 无 trace 无法定位 | 出错后不知道哪步失败 | 懒得写 trace | 每步强制记录，不可跳过 |
| 没有幂等设计 | 重试导致重复写入/扣费 | 忘加 request_id | 消息级 + 操作级双幂等 |
| Agent 循环调用 | 资源耗尽，成本暴增 | A 调 B，B 调 A | 最大调用深度 + 停止条件 |
| 过早并发扩张 | 错误放大，成本飙升 | 没有串行 MVP | 先跑通再扩容 |
| 过度信任 LLM 输出 | 执行了危险 tool call | 没有输出校验 | 参数白名单 + HITL 审批 |
| 忽视数据隐私 | PII 在 Agent 间明文传播 | 缺少脱敏层 | Retriever 输出处脱敏 |
| Verifier 本身不可信 | Verifier 也会出错 | 信任 Verifier 100% | Verifier 也要 trace + 人工抽检 |

---

---
## Part 8 · 安全：输入校验 + 权限矩阵

防止恶意输入穿透到 Worker，并演示最小权限原则。

In [ ]:
ALLOWED_TASKS   = {'summarize', 'extract_risks', 'propose_measures'}
ALLOWED_DOC_IDS = {f'policy-{y}' for y in ['2023','2024','2025']}

@dataclass
class TaskRequest:
    doc_id: str
    task:   str
    metadata: dict = field(default_factory=dict)

    def validate(self):
        errors = []
        if not isinstance(self.doc_id, str) or len(self.doc_id) > 64:
            errors.append(f'doc_id invalid: {self.doc_id!r}')
        if self.doc_id not in ALLOWED_DOC_IDS:
            errors.append(f'doc_id not in whitelist: {self.doc_id!r}')
        if self.task not in ALLOWED_TASKS:
            errors.append(f'task not in whitelist: {self.task!r}')
        if errors:
            raise ValueError('Validation failed: ' + '; '.join(errors))
        return self


PERMISSIONS = {
    'retriever':      {'read_docs': True,  'write_db': False, 'call_llm': False, 'send_alert': False},
    'summarizer':     {'read_docs': True,  'write_db': False, 'call_llm': True,  'send_alert': False},
    'risk_extractor': {'read_docs': True,  'write_db': False, 'call_llm': True,  'send_alert': False},
    'verifier':       {'read_docs': True,  'write_db': False, 'call_llm': True,  'send_alert': False},
    'supervisor':     {'read_docs': True,  'write_db': True,  'call_llm': False, 'send_alert': True},
}

def check_permission(agent, action):
    return PERMISSIONS.get(agent, {}).get(action, False)


print('=== Input Validation Tests ===\n')
test_cases = [
    TaskRequest('policy-2025',  'summarize'),
    TaskRequest('policy-2025',  'DROP TABLE users; --'),
    TaskRequest('secret-admin', 'summarize'),
    TaskRequest('policy-2025',  'delete_all'),
    TaskRequest('policy-2024',  'extract_risks'),
]
for req in test_cases:
    try:
        req.validate()
        print(f'  ALLOWED: doc={req.doc_id!r}  task={req.task!r}')
    except ValueError as e:
        print(f'  BLOCKED: {e}')

print('\n=== Permission Matrix Tests ===\n')
tests = [
    ('retriever',  'read_docs'),
    ('retriever',  'write_db'),
    ('summarizer', 'call_llm'),
    ('summarizer', 'send_alert'),
    ('supervisor', 'write_db'),
    ('supervisor', 'send_alert'),
]
for agent, action in tests:
    ok = check_permission(agent, action)
    status = 'ALLOWED' if ok else 'DENIED '
    print(f'  {status}: {agent:<18} -> {action}')

print('\nSecurity Summary:')
print('  1. All external inputs validated against whitelist')
print('  2. Each Agent has only minimum permissions for its role')
print('  3. Rejection events logged (printed here; production: write to audit DB)')
print('  4. SQL injection blocked at validation layer, never reaches Workers')


## 📖 讲义

# Part 8
## 课堂练习导引

---

# 五阶段练习路径

| 阶段 | 任务 | 验收 |
|------|------|------|
| **1. MVP** | Orchestrator + 3 个串行 Worker，跑通端到端，有 trace 输出 | trace.json 可读 |
| **2. 并行** | Worker 改为 `asyncio.gather` 并行，测量延迟改善 | 延迟 < 串行 60% |
| **3. Verifier** | 加 Verifier + Quality Gate；pass_rate < 0.8 触发 HITL ticket | ticket 正确生成 |
| **4. Supervisor** | 完整 trace 存储 + p95 延迟监控 + Audit Log | trace 可按 job_id 查询 |
| **5. 安全** | Pydantic 入参校验 + 任务白名单 + PII 脱敏 | 恶意输入被拦截并记录 |

**评估标准**

```
功能正确      40%  ── 三页报告、Verifier、HITL 均工作
架构清晰      20%  ── 三层分离，接口有 schema
Trace/监控    15%  ── 每步有记录，有质量门
错误/降级     15%  ── 重试、降级、故障演练
文档与演示    10%  ── README + 架构图 + 演示录屏
```

---

# 推荐演示顺序

**演示时建议按以下顺序展示，从正常到异常：**

1. **正常路径**（pass_rate=0.9）
   - 展示完整 trace 甘特图
   - 指出并行节省的时间

2. **低质量触发 HITL**（pass_rate=0.6）
   - 展示 HITL ticket 内容
   - 展示审核界面（或 JSON 模拟）

3. **Worker 故障 → 降级**
   - 手动让 Reranker 抛异常
   - 展示降级路径和日志

4. **Trace 回放**
   - 从 trace.json 重现某次失败
   - 定位到具体的失败 step

---

# 推荐工具栈

<div class="columns">
<div>

### 框架
- **LangGraph** — 分层 Agent 状态机，支持 checkpointing
- **CrewAI** — Role-based 多 Agent
- **AutoGen** — 微软多 Agent 框架
- **Prefect / Temporal** — 生产工作流编排

### 向量数据库
- **FAISS** — 本地开发
- **Qdrant / Weaviate** — 生产推荐
- **Pinecone** — Serverless 选项

</div>
<div>

### 可观测性
- **Langfuse** — LLM trace & eval（开源可自建）
- **LangSmith** — LangChain 专用
- **OpenTelemetry** — 标准分布式追踪
- **Prometheus + Grafana** — Metrics

### NLI / Verifier
- `cross-encoder/nli-deberta-v3-small` — 轻量 NLI
- **RAGAS** — RAG 质量评估框架
- **TruLens** — LLM 应用评估

### 消息队列
- **Redis Streams** — 入门首选
- **Kafka** — 高吞吐生产环境

</div>
</div>

---

# 课程小结

**分层 Multi-Agent + Supervision = 可扩展 + 可维护 + 可治理**

```mermaid
flowchart TB
    Supervisor["Supervisor<br/>质量、合规、HITL、审计"]
    Orchestrator["Orchestrator<br/>计划、并行分派、汇总"]
    Workers["Workers<br/>专注单一技能：检索 / 摘要 / 校验"]

    Supervisor --> Orchestrator --> Workers
```

**三条铁律**
1. **每层只做自己的事** — Worker 不计划，Orchestrator 不检索
2. **每步必有 trace** — 无 trace = 盲飞，出事无法定位
3. **质量门 + HITL** — 自动校验 + 人工兜底，不纯自动化高风险决策

**演进路径**
```mermaid
flowchart LR
    A[单体 MVP] --> B[串行三层架构<br/>有 trace]
    B --> C[并行 Workers<br/>asyncio.gather]
    C --> D[Verifier + Quality Gate]
    D --> E[Supervisor + HITL]
    E --> F[容器化 + 压测 + 监控]
```

---

---

# Q & A

## 有什么问题？

<br>

**课后作业**：完成练习阶段 1–3
- 阶段 1：串行三层架构，有 trace 输出
- 阶段 2：改为并行，延迟 < 串行 60%
- 阶段 3：加 Verifier + HITL ticket

提交：代码仓库 + `trace.json` 样例 + 延迟对比截图

<br>

*下一课：Agent 评估框架与持续优化*


---
## 课程小结

| 组件 | 职责 | 关键实现 |
|------|------|----------|
| **Worker Agents** | 专注单一技能 | 小函数，无副作用，可独立测试 |
| **Orchestrator** | 计划 + 分派 + 汇总 | `asyncio.gather` 并行，不做具体任务 |
| **Verifier** | 关键断言 NLI 校验 | pass_rate >= 0.80 才通过 |
| **Supervisor** | 质量门 + HITL + 审计 | trace 每步必记，HITL 分级触发 |
| **TraceStore** | 全链路可观测 | job_id 串联，延迟甘特图 |
| **安全层** | 输入校验 + 权限矩阵 | 白名单优先，最小权限原则 |

### 三条铁律

1. **每层只做自己的事** — Worker 不计划，Orchestrator 不检索
2. **每步必有 trace** — 无 trace = 盲飞，出事无法定位
3. **质量门 + HITL** — 自动校验 + 人工兜底，高风险不能纯自动化

---
*下一步：将模拟工具替换为真实 LLM API + 向量数据库，接入 Redis 消息队列*
